# Taller 2 (AWS Academy): ETL con Apache Spark y Delta Lake en EMR

Implementarás el mismo flujo ETL Medallion (Bronze → Silver → Gold) del Taller Databricks,  
pero sobre **Amazon Web Services** usando **EMR** (Elastic MapReduce) como motor de Spark  
y **S3** como sistema de almacenamiento del data lake.

## Diferencias clave respecto a Databricks
| Concepto | Databricks | AWS EMR |
|---|---|---|
| Almacenamiento | DBFS (`/tmp/...`) | S3 (`s3://bucket/...`) |
| Operaciones de archivos | `dbutils.fs` | `boto3` o AWS CLI (`aws s3`) |
| Visualización en notebook | `display(df)` | `df.show()` / `df.limit(n).toPandas()` |
| Delta Lake | Pre-instalado | Configurado al crear el cluster |
| SparkSession | Pre-inicializada | Pre-inicializada (kernel PySpark) |

## ¿Qué es AWS Academy?

AWS Academy es un programa educativo de Amazon que otorga acceso gratuito a servicios de AWS  
a estudiantes a través de sus instituciones. Incluye un **Learner Lab** con créditos de ~$100 USD  
y acceso a los servicios necesarios para este taller: **S3**, **EMR** y **EMR Notebooks**.

---
## 0. Configuración del entorno en AWS Academy

Sigue estos pasos **una sola vez** antes de ejecutar el notebook.

---

### Paso 1 — Acceder al Learner Lab de AWS Academy

Tu institución debe estar inscrita en AWS Academy. Si no has recibido la invitación, consulta con tu docente.

**Windows / macOS / Linux:**
1. Revisa tu correo institucional: deberías haber recibido una invitación a **AWS Academy**
2. Acepta la invitación y crea tu cuenta en https://www.awsacademy.com/
3. Inicia sesión → navega a tu curso asignado
4. Abre el módulo **"Learner Lab"** (o **"AWS Academy Learner Lab"**)
5. Haz clic en **"Start Lab"** — espera 1-2 minutos hasta que el indicador cambie a verde (●)
6. Haz clic en **"AWS"** (enlace verde) para abrir la consola de AWS en una nueva pestaña

> Monitorea tu saldo restante en el panel del Learner Lab. El saldo inicial es ~$100 USD.  
> El lab se pausa automáticamente tras un período de inactividad para conservar créditos.

---

### Paso 2 — Crear un bucket en S3

S3 es el equivalente al DBFS de Databricks: el almacenamiento distribuido del data lake.

1. En la consola de AWS, escribe **"S3"** en la barra de búsqueda y selecciónalo
2. Haz clic en **"Create bucket"**
3. **Bucket name:** elige un nombre único globalmente, por ejemplo `taller2-spark-tunombre`  
   *(los nombres de bucket S3 son únicos en todo AWS)*
4. **AWS Region:** usa la región asignada por el Learner Lab (generalmente `us-east-1`)
5. Deja **Block all public access** activado (opción por defecto)
6. Haz clic en **"Create bucket"**

> Anota el nombre exacto del bucket. Lo usarás en la variable `BUCKET_NAME` del notebook.

---

### Paso 3 — Crear el cluster de EMR con Spark y Delta Lake

EMR es el servicio de Spark administrado de AWS, equivalente a Dataproc en GCP.

1. En la consola, busca **"EMR"** y selecciónalo
2. En el panel izquierdo, selecciona **"Clusters"** → haz clic en **"Create cluster"**
3. Configura las opciones principales:
   - **Cluster name:** `taller2-cluster`
   - **Amazon EMR release:** `emr-6.15.0` o la versión más reciente disponible (6.x o superior)
   - **Application bundle:** selecciona **"Spark"** (incluye Spark + JupyterEnterpriseGateway)
4. En **"Instance configuration"**:
   - **Primary node:** `m5.xlarge` (suficiente para el taller)
   - **Core nodes:** cambia la cantidad a `0` (solo nodo principal, más económico)
5. En **"Software settings"**, pega la siguiente configuración JSON para activar Delta Lake:

```json
[
  {
    "Classification": "spark-defaults",
    "Properties": {
      "spark.jars.packages": "io.delta:delta-core_2.12:2.4.0",
      "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
      "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    }
  }
]
```

6. En **"Cluster logs"**: activa **"Publish cluster-specific logs to Amazon S3"** y selecciona tu bucket
7. En **"Security configuration"**: en el Learner Lab generalmente se puede dejar el EC2 key pair en blanco
8. Haz clic en **"Create cluster"** y espera 5-10 minutos hasta que el estado sea **"Waiting"** (color verde)

> **Costo estimado:** un cluster `m5.xlarge` consume aproximadamente $0.20 USD/hora de tus créditos.  
> Recuerda **terminarlo** (Terminate) cuando termines el taller para no gastar créditos innecesariamente.

---

### Paso 4 — Crear un EMR Notebook y conectarlo al cluster

EMR Notebooks (parte de EMR Studio) es el equivalente al workspace de notebooks de Databricks.

1. En el panel de EMR, selecciona **"Notebooks"** en el menú izquierdo
2. Haz clic en **"Create notebook"**
3. Configura:
   - **Notebook name:** `taller2-notebook`
   - **Cluster:** selecciona **"Choose an existing cluster"** → selecciona `taller2-cluster`
   - **AWS Service Role:** deja la opción por defecto del Learner Lab
   - **Notebook location:** selecciona tu bucket S3
4. Haz clic en **"Create notebook"**
5. Una vez creado (estado **"Ready"**), haz clic en **"Open in JupyterLab"**

---

### Paso 5 — Importar este notebook y configurar el kernel

1. En JupyterLab, ve a **File → Upload Files** y sube el archivo `guia_aws_academy.ipynb`
2. Abre el notebook
3. **Verifica que el kernel sea PySpark** (esquina superior derecha del notebook)
   - Si no dice PySpark, haz clic → selecciona **PySpark**
4. En la primera celda de código, reemplaza `TU_BUCKET` con el nombre de tu bucket S3

> En EMR Notebooks con kernel PySpark, `spark` y `sc` están disponibles automáticamente.  
> No necesitas crear SparkSession ni instalar nada adicional.

---
## Arquitectura del taller

```
  DATOS SINTETICOS
  (generados con Spark)
          |
          v
 +----------------+
 |    BRONZE      |  s3://bucket/taller2/bronze/
 |   Delta Lake   |  Datos crudos, sin transformar
 +----------------+
          |
          v
 +----------------+
 |    SILVER      |  s3://bucket/taller2/silver/
 |   Delta Lake   |  Datos limpios y enriquecidos
 +----------------+
          |
          v
 +----------------+
 |     GOLD       |  s3://bucket/taller2/gold/
 |   Delta Lake   |  Metricas para consumo analitico
 +----------------+
```

**Almacenamiento:** Amazon S3  
**Formato:** Delta Lake (Parquet + transaction log)  
**Motor de procesamiento:** Apache Spark en Amazon EMR

In [1]:
# =============================================================
# CONFIGURACION — ACTUALIZA ESTE VALOR ANTES DE CONTINUAR
# =============================================================

BUCKET_NAME = "taller2-spark-samuel-acosta"  # <-- reemplaza con el nombre de tu bucket S3

# Rutas en S3 (equivalentes al DBFS de Databricks)
BASE_PATH   = f"s3://{BUCKET_NAME}/taller2"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH   = f"{BASE_PATH}/gold"

# Limpiar ejecuciones anteriores
# aws s3 rm reemplaza a dbutils.fs.rm en el contexto de S3
import subprocess
subprocess.run(
    ["aws", "s3", "rm", f"s3://{BUCKET_NAME}/taller2/", "--recursive"],
    capture_output=True
)
print("Directorio anterior eliminado (si existia).")

print(f"\nRutas configuradas:")
print(f"  Bronze : {BRONZE_PATH}")
print(f"  Silver : {SILVER_PATH}")
print(f"  Gold   : {GOLD_PATH}")
print(f"\nSpark version: {spark.version}")

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1776710655295_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Directorio anterior eliminado (si existia).

Rutas configuradas:
  Bronze : s3://taller2-spark-samuel-acosta/taller2/bronze
  Silver : s3://taller2-spark-samuel-acosta/taller2/silver
  Gold   : s3://taller2-spark-samuel-acosta/taller2/gold

Spark version: 3.4.1-amzn-2

---
## Parte 1 — Bronze: Ingesta de datos crudos

Generamos 1.000 transacciones de ventas sintéticas y las guardamos como **Delta Lake en S3**.

In [2]:
# =============================================================
# 1.1 — GENERAR DATOS SINTETICOS
# =============================================================

from pyspark.sql import functions as F

SEED = 42

arr_categorias = F.array([F.lit(c) for c in ["Electronica", "Muebles", "Ropa", "Alimentos", "Libros"]])
arr_ciudades   = F.array([F.lit(c) for c in ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena"]])
arr_pagos      = F.array([F.lit(p) for p in ["tarjeta_credito", "tarjeta_debito", "efectivo", "transferencia"]])
arr_productos  = F.array([F.lit(p) for p in [
    "Laptop Pro", "Silla Ergonomica", "Camiseta Algodon", "Arroz Premium",
    "Python Avanzado", "Monitor 4K", "Escritorio Pie", "Jean Clasico",
    "Aceite Oliva", "SQL para Todos", "Auriculares BT", "Estante Madera",
    "Zapatos Cuero", "Leche Entera", "Historia IA"
]])

df_bronze_raw = (
    spark.range(1, 1001)
    .select(
        F.col("id"),
        F.rand(SEED).alias("r1"),
        F.rand(SEED + 1).alias("r2"),
        F.rand(SEED + 2).alias("r3"),
        F.rand(SEED + 3).alias("r4"),
        F.rand(SEED + 4).alias("r5"),
        F.rand(SEED + 5).alias("r6"),
        F.rand(SEED + 6).alias("r7"),
    )
    .select(
        F.concat(F.lit("TXN-"), F.lpad(F.col("id").cast("string"), 5, "0")).alias("transaction_id"),
        F.concat(F.lit("C"), F.lpad(((F.col("r1") * 200).cast("int") + 1).cast("string"), 4, "0")).alias("customer_id"),
        arr_productos[(F.col("r2") * 15).cast("int")].alias("product_name"),
        arr_categorias[(F.col("r3") * 5).cast("int")].alias("category"),
        ((F.col("r4") * 9).cast("int") + 1).alias("quantity"),
        F.when(F.col("r5") < 0.03, F.lit(-99.0))
         .otherwise(F.round(F.col("r5") * 1995 + 5, 2)).alias("unit_price"),
        F.date_add(F.lit("2024-01-01").cast("date"), (F.col("r6") * 365).cast("int")).alias("transaction_date"),
        arr_ciudades[(F.col("r1") * 5).cast("int")].alias("store_city"),
        arr_pagos[(F.col("r2") * 4).cast("int")].alias("payment_method"),
        F.when(F.col("r7") > 0.9, F.lit(None).cast("double"))
         .otherwise((F.col("r4") * 4).cast("int") * 0.05).alias("discount_pct"),
    )
)

print(f"Registros generados: {df_bronze_raw.count():,}")
df_bronze_raw.printSchema()
df_bronze_raw.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Registros generados: 1,000
root
 |-- transaction_id: string (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- store_city: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- discount_pct: double (nullable = true)

+--------------+-----------+--------------+---------+--------+----------+----------------+------------+---------------+-------------------+
|transaction_id|customer_id|product_name  |category |quantity|unit_price|transaction_date|store_city  |payment_method |discount_pct       |
+--------------+-----------+--------------+---------+--------+----------+----------------+------------+---------------+-------------------+
|TXN-00001     |C0124      |Zapatos Cuero |Alimentos|8       |85.01     |2024-09-02      |Barranquilla|transfere

In [3]:
# =============================================================
# 1.2 — GUARDAR BRONZE EN S3 COMO DELTA LAKE
# Delta Lake escribe en S3 exactamente igual que en DBFS.
# El transaction log (_delta_log/) se almacena en S3.
# =============================================================

(
    df_bronze_raw
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/ventas/")
)

print(f"Tabla Bronze guardada en: {BRONZE_PATH}/ventas/")

# Ver archivos en S3 (aws s3 ls reemplaza a dbutils.fs.ls)
result = subprocess.run(
    ["aws", "s3", "ls", f"s3://{BUCKET_NAME}/taller2/bronze/ventas/"],
    capture_output=True, text=True
)
print("\nArchivos en Bronze (S3):")
print(result.stdout)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Tabla Bronze guardada en: s3://taller2-spark-samuel-acosta/taller2/bronze/ventas/

Archivos en Bronze (S3):
                           PRE _delta_log/
2026-04-20 19:03:33          0 _delta_log_$folder$
2026-04-20 19:03:38      12797 part-00000-57e96e69-8b25-40d6-926e-fd131d4f874e-c000.snappy.parquet
2026-04-20 19:03:38      12774 part-00001-de58a3a3-aaf8-4987-86d1-1a8585787c0b-c000.snappy.parquet

In [5]:
# =============================================================
# 1.3 — EXPLORAR BRONZE
# En EMR Notebooks usamos .show() en lugar de display()
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

print(f"Total registros  : {df_bronze.count():,}")
print(f"Total columnas   : {len(df_bronze.columns)}")

print("\n=== Nulos por columna ===")
df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
]).show()

print(f"Precios invalidos (< 0): {df_bronze.filter(F.col('unit_price') < 0).count()}")

print("\n=== Distribucion por categoria ===")
df_bronze.groupBy("category").count().orderBy("count", ascending=False).show()

# Muestra en formato tabla Pandas (para datasets pequenos)
print("\n=== Muestra (primeras 5 filas) ===")
df_bronze.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total registros  : 1,000
Total columnas   : 10

=== Nulos por columna ===
+--------------+-----------+------------+--------+--------+----------+----------------+----------+--------------+------------+
|transaction_id|customer_id|product_name|category|quantity|unit_price|transaction_date|store_city|payment_method|discount_pct|
+--------------+-----------+------------+--------+--------+----------+----------------+----------+--------------+------------+
|             0|          0|           0|       0|       0|         0|               0|         0|             0|         100|
+--------------+-----------+------------+--------+--------+----------+----------------+----------+--------------+------------+

Precios invalidos (< 0): 34

=== Distribucion por categoria ===
+-----------+-----+
|   category|count|
+-----------+-----+
|Electronica|  215|
|       Ropa|  208|
|     Libros|  206|
|  Alimentos|  186|
|    Muebles|  185|
+-----------+-----+


=== Muestra (primeras 5 filas) ===
+--------

---
## Parte 2 — Silver: Limpieza y transformación

In [6]:
# =============================================================
# 2.1 — TRANSFORMACION SILVER
# Leer Bronze desde S3 → limpiar → enriquecer → guardar Silver en S3
# =============================================================

df_bronze = spark.read.format("delta").load(f"{BRONZE_PATH}/ventas/")

df_silver = (
    df_bronze
    .filter(F.col("unit_price") > 0)
    .fillna({"discount_pct": 0.0})
    .withColumn("total_bruto",    F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("total_descuento",F.round(F.col("total_bruto") * F.col("discount_pct"), 2))
    .withColumn("total_neto",     F.round(F.col("total_bruto") - F.col("total_descuento"), 2))
    .withColumn("year",           F.year("transaction_date"))
    .withColumn("month",          F.month("transaction_date"))
    .withColumn("month_name",     F.date_format("transaction_date", "MMMM"))
    .withColumn("day_of_week",    F.date_format("transaction_date", "EEEE"))
    .withColumn("processed_at",   F.current_timestamp())
)

print(f"Bronze  : {df_bronze.count():,} registros")
print(f"Silver  : {df_silver.count():,} registros")
print(f"Eliminados: {df_bronze.count() - df_silver.count():,}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .save(f"{SILVER_PATH}/ventas/")
)

print(f"\nSilver guardada en: {SILVER_PATH}/ventas/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Bronze  : 1,000 registros
Silver  : 966 registros
Eliminados: 34

Silver guardada en: s3://taller2-spark-samuel-acosta/taller2/silver/ventas/

In [7]:
# =============================================================
# 2.2 — VERIFICAR SILVER
# =============================================================

df_silver_check = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

print("=== Nulos en columnas criticas (deben ser 0) ===")
df_silver_check.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["unit_price", "discount_pct", "total_neto"]
]).show()

print("=== Muestra de datos Silver ===")
df_silver_check.select(
    "transaction_id", "customer_id", "category",
    "quantity", "unit_price", "discount_pct", "total_neto",
    "year", "month"
).show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Nulos en columnas criticas (deben ser 0) ===
+----------+------------+----------+
|unit_price|discount_pct|total_neto|
+----------+------------+----------+
|         0|           0|         0|
+----------+------------+----------+

=== Muestra de datos Silver ===
+--------------+-----------+-----------+--------+----------+-------------------+----------+----+-----+
|transaction_id|customer_id|   category|quantity|unit_price|       discount_pct|total_neto|year|month|
+--------------+-----------+-----------+--------+----------+-------------------+----------+----+-----+
|     TXN-00511|      C0131|       Ropa|       6|    738.58|                0.1|   3988.33|2024|    4|
|     TXN-00512|      C0135|       Ropa|       6|   1665.36|                0.1|   8992.94|2024|    4|
|     TXN-00525|      C0023|    Muebles|       4|    621.33|               0.05|   2361.05|2024|    4|
|     TXN-00557|      C0182|    Muebles|       6|   1855.93|                0.1|  10022.02|2024|    4|
|     TXN-00

---
## Parte 3 — Gold: Métricas y agregaciones

In [8]:
# =============================================================
# 3.1 — VENTAS POR CATEGORIA Y MES
# =============================================================

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

gold_cat_mes = (
    df_silver
    .groupBy("year", "month", "month_name", "category")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.sum("quantity").alias("unidades_vendidas"),
        F.round(F.sum("total_bruto"), 2).alias("ingresos_brutos"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("unit_price"), 2).alias("precio_promedio"),
    )
    .orderBy("year", "month", F.col("ingresos_netos").desc())
)

gold_cat_mes.show(15)
gold_cat_mes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/ventas_categoria_mes/")
print(f"Guardado en: {GOLD_PATH}/ventas_categoria_mes/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----+----------+-----------+-----------------+-----------------+---------------+--------------+---------------+
|year|month|month_name|   category|num_transacciones|unidades_vendidas|ingresos_brutos|ingresos_netos|precio_promedio|
+----+-----+----------+-----------+-----------------+-----------------+---------------+--------------+---------------+
|2024|    1|   January|     Libros|               17|               89|       84023.39|      74890.22|         953.72|
|2024|    1|   January|       Ropa|               16|               77|       77971.72|      70002.18|         961.93|
|2024|    1|   January|Electronica|               18|               78|       73603.83|      67775.09|         917.88|
|2024|    1|   January|    Muebles|               14|               60|       72068.62|      65522.47|        1086.65|
|2024|    1|   January|  Alimentos|               14|               74|       60345.42|      55679.81|         790.04|
|2024|    2|  February|       Ropa|             

In [9]:
# =============================================================
# 3.2 — TOP CLIENTES + SEGMENTACION
# =============================================================

gold_clientes = (
    df_silver
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("num_compras"),
        F.round(F.sum("total_neto"), 2).alias("gasto_total"),
        F.round(F.avg("total_neto"), 2).alias("gasto_promedio"),
        F.countDistinct("category").alias("categorias_distintas"),
    )
    .withColumn(
        "segmento",
        F.when(F.col("gasto_total") >= 5000, "VIP")
         .when(F.col("gasto_total") >= 2000, "Premium")
         .when(F.col("gasto_total") >= 500,  "Regular")
         .otherwise("Ocasional")
    )
    .orderBy(F.col("gasto_total").desc())
)

print("=== Top 15 clientes ===")
gold_clientes.show(15)

print("=== Distribucion de segmentos ===")
gold_clientes.groupBy("segmento").count().orderBy("count", ascending=False).show()

gold_clientes.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/top_clientes/")
print(f"Guardado en: {GOLD_PATH}/top_clientes/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Top 15 clientes ===
+-----------+-----------+-----------+--------------+--------------------+--------+
|customer_id|num_compras|gasto_total|gasto_promedio|categorias_distintas|segmento|
+-----------+-----------+-----------+--------------+--------------------+--------+
|      C0200|         13|   89411.26|       6877.79|                   5|     VIP|
|      C0044|         10|   70668.57|       7066.86|                   4|     VIP|
|      C0149|          9|   67869.59|       7541.07|                   4|     VIP|
|      C0130|         13|   62815.21|       4831.94|                   5|     VIP|
|      C0133|          9|   61700.89|       6855.65|                   4|     VIP|
|      C0100|          9|   54385.39|       6042.82|                   5|     VIP|
|      C0127|          8|   54207.71|       6775.96|                   4|     VIP|
|      C0155|          9|   52881.79|       5875.75|                   3|     VIP|
|      C0016|          6|   52308.99|       8718.17|           

In [10]:
# =============================================================
# 3.3 — ANALISIS POR METODO DE PAGO (Spark SQL)
# =============================================================

df_silver.createOrReplaceTempView("silver_ventas")

gold_pagos = spark.sql("""
    SELECT
        payment_method,
        COUNT(*)                           AS num_transacciones,
        ROUND(SUM(total_neto), 2)          AS ingresos_netos,
        ROUND(AVG(total_neto), 2)          AS ticket_promedio,
        ROUND(AVG(discount_pct) * 100, 1)  AS descuento_promedio_pct
    FROM silver_ventas
    GROUP BY payment_method
    ORDER BY ingresos_netos DESC
""")

gold_pagos.show()
gold_pagos.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/metodos_pago/")
print(f"Guardado en: {GOLD_PATH}/metodos_pago/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-----------------+--------------+---------------+----------------------+
| payment_method|num_transacciones|ingresos_netos|ticket_promedio|descuento_promedio_pct|
+---------------+-----------------+--------------+---------------+----------------------+
|  transferencia|              263|    1280346.03|        4868.24|                   6.7|
| tarjeta_debito|              231|    1162463.82|        5032.31|                   6.5|
|tarjeta_credito|              235|     1078160.7|        4587.92|                   6.6|
|       efectivo|              237|    1068834.03|        4509.85|                   7.1|
+---------------+-----------------+--------------+---------------+----------------------+

Guardado en: s3://taller2-spark-samuel-acosta/taller2/gold/metodos_pago/

---
## Parte 4 — Delta Lake: Capacidades avanzadas

Delta Lake es un **formato de tabla abierto** que funciona de forma idéntica sobre S3 que sobre DBFS.  
El transaction log (`_delta_log/`) se almacena directamente en el bucket S3.

In [12]:
# =============================================================
# 4.1 — TRANSACTION LOG: DESCRIBE HISTORY
# =============================================================

import subprocess
from delta.tables import DeltaTable

# Cargamos el objeto DeltaTable
silver_dt = DeltaTable.forPath(spark, f"{SILVER_PATH}/ventas/")

print("=== Historial de la tabla Silver ===")

# Corregimos el acceso a numOutputRows usando el prefijo operationMetrics
silver_dt.history().select(
    "version", 
    "timestamp", 
    "operation", 
    "operationMetrics.numOutputRows" # Sintaxis de punto para acceder al mapa
).show(truncate=False)

# Ver el transaction log en S3 (verificamos archivos .json y .checkpoint)
result = subprocess.run(
    ["aws", "s3", "ls", f"s3://{BUCKET_NAME}/taller2/silver/ventas/_delta_log/"],
    capture_output=True, text=True
)

print("\nArchivos del transaction log en S3:")
print(result.stdout)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Historial de la tabla Silver ===
+-------+-------------------+---------+-------------+
|version|timestamp          |operation|numOutputRows|
+-------+-------------------+---------+-------------+
|0      |2026-04-20 19:10:31|WRITE    |966          |
+-------+-------------------+---------+-------------+


Archivos del transaction log en S3:
2026-04-20 19:10:31      38241 00000000000000000000.json

In [13]:
# =============================================================
# 4.2 — CREAR NUEVA VERSION CON APPEND
# =============================================================

df_nuevos = df_silver.limit(50).withColumn("processed_at", F.current_timestamp())

(
    df_nuevos
    .write
    .format("delta")
    .mode("append")
    .save(f"{SILVER_PATH}/ventas/")
)

print("Append completado. Versiones disponibles:")
silver_dt.history().select("version", "timestamp", "operation").show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Append completado. Versiones disponibles:
+-------+-------------------+---------+
|version|          timestamp|operation|
+-------+-------------------+---------+
|      1|2026-04-20 19:24:22|    WRITE|
|      0|2026-04-20 19:10:31|    WRITE|
+-------+-------------------+---------+

In [14]:
# =============================================================
# 4.3 — TIME TRAVEL: LEER VERSION ANTERIOR
# Funciona igual en S3 que en DBFS gracias al transaction log
# =============================================================

df_actual = spark.read.format("delta").load(f"{SILVER_PATH}/ventas/")

df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(f"{SILVER_PATH}/ventas/")
)

print(f"Version actual  : {df_actual.count():,} registros")
print(f"Version 0       : {df_v0.count():,} registros")
print(f"Diferencia      : {df_actual.count() - df_v0.count():,} registros")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Version actual  : 1,016 registros
Version 0       : 966 registros
Diferencia      : 50 registros

In [15]:
# =============================================================
# 4.4 — OPTIMIZE: COMPACTAR ARCHIVOS EN S3
# =============================================================

spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}/ventas/`")

print("OPTIMIZE completado.")
silver_dt.history().select("version", "operation").show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

OPTIMIZE completado.
+-------+---------+
|version|operation|
+-------+---------+
|      2| OPTIMIZE|
|      1|    WRITE|
|      0|    WRITE|
+-------+---------+

---
## Parte 5 — Ejercicios integradores

Los mismos ejercicios del Taller Databricks, ahora ejecutándose sobre AWS.

**E1** — Producto más vendido (en unidades) por categoría.

In [16]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Definimos la ventana: particionamos por categoría y ordenamos por unidades vendidas
w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())

print("=== E1: Producto Top por Categoría ===")
(
    df_silver.groupBy("category", "product_name")
    .agg(F.sum("quantity").alias("total_unidades"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy("category")
    .show()
)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== E1: Producto Top por Categor?a ===
+-----------+--------------+--------------+
|   category|  product_name|total_unidades|
+-----------+--------------+--------------+
|  Alimentos|Escritorio Pie|           107|
|Electronica|  Jean Clasico|           122|
|     Libros|Auriculares BT|           107|
|    Muebles|  Jean Clasico|            86|
|    Muebles|   Historia IA|            86|
|       Ropa|SQL para Todos|           132|
+-----------+--------------+--------------+

**E2** — Ingresos netos y tasa de descuento promedio por ciudad (`store_city`). Ordena de mayor a menor ingreso.

In [17]:
import pyspark.sql.functions as F

print("=== E2: Descuentos e Ingresos por Ciudad ===")
(
    df_silver.groupBy("store_city")
    .agg(
        F.round(F.avg("discount_pct") * 100, 1).alias("descuento_prom_pct"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos")
    )
    .orderBy(F.col("ingresos_netos").desc())
    .show()
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== E2: Descuentos e Ingresos por Ciudad ===
+------------+------------------+--------------+
|  store_city|descuento_prom_pct|ingresos_netos|
+------------+------------------+--------------+
|Barranquilla|               6.7|    1081594.22|
|        Cali|               6.0|     975427.03|
|    Medellin|               7.1|     955402.04|
|   Cartagena|               7.2|     919730.62|
|      Bogota|               6.9|     909475.75|
+------------+------------------+--------------+

**E3** — Tabla Gold `tendencia_semanal` con: semana del año, número de transacciones, ingresos netos y ticket promedio. Guárdala en `{GOLD_PATH}/tendencia_semanal/` como Delta.

In [18]:
import pyspark.sql.functions as F

print("=== E3: Generando Capa Gold (Tendencia Semanal) ===")

gold_semanal = (
    df_silver
    .withColumn("week", F.weekofyear("transaction_date"))
    .groupBy("year", "week")
    .agg(
        F.count("transaction_id").alias("num_transacciones"),
        F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
        F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
    )
    .orderBy("year", "week")
)

# Guardar en S3 en formato Delta
gold_semanal.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/tendencia_semanal/")

# Mostrar resultado
gold_semanal.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== E3: Generando Capa Gold (Tendencia Semanal) ===
+----+----+-----------------+--------------+---------------+
|year|week|num_transacciones|ingresos_netos|ticket_promedio|
+----+----+-----------------+--------------+---------------+
|2024|   1|               21|     113995.49|        5428.36|
|2024|   2|               20|      85089.51|        4254.48|
|2024|   3|               21|      87432.04|        4163.43|
|2024|   4|               17|      60351.16|        3550.07|
|2024|   5|               14|      63427.49|        4530.54|
|2024|   6|               20|      98310.32|        4915.52|
|2024|   7|               28|     131178.36|        4684.94|
|2024|   8|               21|     106304.76|        5062.13|
|2024|   9|               26|      86813.99|         3339.0|
|2024|  10|               18|     107351.78|        5963.99|
|2024|  11|               17|      97119.99|        5712.94|
|2024|  12|               20|       81642.7|        4082.13|
|2024|  13|               13|    

---
## Soluciones de referencia

In [ ]:
# --- E1 ---
# from pyspark.sql import Window
# w = Window.partitionBy("category").orderBy(F.col("total_unidades").desc())
# (
#     df_silver.groupBy("category", "product_name")
#     .agg(F.sum("quantity").alias("total_unidades"))
#     .withColumn("rank", F.rank().over(w))
#     .filter(F.col("rank") == 1).drop("rank")
#     .orderBy("category").show()
# )

# --- E2 ---
# (
#     df_silver.groupBy("store_city")
#     .agg(
#         F.round(F.avg("discount_pct") * 100, 1).alias("descuento_prom_pct"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos")
#     )
#     .orderBy(F.col("ingresos_netos").desc()).show()
# )

# --- E3 ---
# gold_semanal = (
#     df_silver
#     .withColumn("week", F.weekofyear("transaction_date"))
#     .groupBy("year", "week")
#     .agg(
#         F.count("transaction_id").alias("num_transacciones"),
#         F.round(F.sum("total_neto"), 2).alias("ingresos_netos"),
#         F.round(F.avg("total_neto"), 2).alias("ticket_promedio")
#     )
#     .orderBy("year", "week")
# )
# gold_semanal.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/tendencia_semanal/")
# gold_semanal.show()

---
## Limpieza de recursos

**Importante:** para preservar tus créditos del Learner Lab, termina el cluster cuando termines el taller.

In [19]:
# Eliminar datos del taller del bucket S3
result = subprocess.run(
    ["aws", "s3", "rm", f"s3://{BUCKET_NAME}/taller2/", "--recursive"],
    capture_output=True, text=True
)
print(f"Datos eliminados de s3://{BUCKET_NAME}/taller2/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Datos eliminados de s3://taller2-spark-samuel-acosta/taller2/

### Terminar el cluster de EMR

1. Ve a la consola de AWS → **EMR** → **Clusters**
2. Selecciona `taller2-cluster`
3. Haz clic en **"Terminate"** y confirma

> Un cluster en estado **"Waiting"** sigue consumiendo créditos aunque no esté procesando datos.

### Pausar el Learner Lab (si ya no vas a continuar)

1. Regresa al portal de AWS Academy
2. Haz clic en **"End Lab"** para pausar el entorno y detener el consumo de créditos